# Checkpoint IV · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_4/03_examen_solutions.ipynb)

La solution de la question de code (CP4.9), exécutée et comparée à PyTorch, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · La question de code

In [ ]:
# Data and tools of the code question CP4.9 (run this cell first)
import numpy as np

# 2 048 images of Fashion-MNIST, flattened to 784 values in [0, 1] (float64), and their labels 0-9
X_F, Y_F = wb.datasets.load_fashion_mnist("train", n=2048, flatten=True, normalize=True, seed=49)
X_F = X_F.astype(np.float64)
LR, BATCH, MAX_NORM = 1e-2, 100, 1.0          # the settings of the try (CP4.9 d)


def init_params(seed=49):
    """The starting parameters [W, b] of the softmax regression 784 -> 10 (new arrays at each call)."""
    rng = np.random.default_rng(seed)
    return [rng.normal(0.0, 0.01, size=(784, 10)), np.zeros(10)]


def loss_and_grads(params, X, y):
    """(mean cross-entropy of the mini-batch, [dW, db]) of the softmax regression z = X @ W + b."""
    W, b = params
    z = X @ W + b
    z = z - z.max(axis=1, keepdims=True)                       # log-sum-exp trick (ch. 17)
    p = np.exp(z)
    p /= p.sum(axis=1, keepdims=True)
    n = len(y)
    loss = -np.log(p[np.arange(n), y]).mean()
    d = p.copy()
    d[np.arange(n), y] -= 1.0                                  # dL/dz = (p - one_hot(y)) / n  (ch. 18)
    d /= n
    return float(loss), [X.T @ d, d.sum(axis=0)]


def accuracy(params, X, y):
    """Share of the images whose largest logit is the right class."""
    return float(((X @ params[0] + params[1]).argmax(axis=1) == y).mean())


_ORIGINALS = {"X_F": X_F.copy(), "Y_F": Y_F.copy()}         # part B restores them if a try changed them
print("X_F:", X_F.shape, "· Y_F:", Y_F.shape, "· starting loss:", round(loss_and_grads(init_params(), X_F, Y_F)[0], 4))

### CP4.9 — Écrire de mémoire une epoch d'entraînement avec AdamW et clipping, en NumPy 🔨 ★★ ⏱️ 15 min · 2 points
**Objectif :** écrire sans aide le cœur d'une boucle d'entraînement : clipping par la norme globale, un pas d'AdamW, une epoch en mini-batches.

Avec NumPy seulement. Une régression softmax (784 → 10) sur les 2 048 images de `X_F` et leurs labels `Y_F` ; ses paramètres sont la liste `params = [W, b]` (`init_params()` en donne une nouvelle à chaque appel). `loss_and_grads(params, X, y)` rend la cross-entropy moyenne du mini-batch et la liste `[dW, db]` de ses gradients ; `accuracy(params, X, y)` rend l'accuracy.

a) `clip_by_global_norm(grads, max_norm)` rend `(clipped, norm)` : `norm`, la norme globale (racine de la somme des carrés de **tous** les gradients) ; `clipped`, une liste de **nouveaux** tableaux, les gradients multipliés par $\min\left(1, \frac{\text{max\_norm}}{\text{norm} + 10^{-6}}\right)$. Elle ne modifie pas `grads`, à la différence de `clip_grad_norm` (ch. 19) et de `torch.nn.utils.clip_grad_norm_`, qui modifient les gradients sur place. **[0,5]**
b) `adamw_step(params, grads, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)` fait un pas d'AdamW **sur place**, avec les conventions de `torch.optim.AdamW` : $t \leftarrow t + 1$ ; $w \leftarrow w\,(1 - \eta\lambda)$ ; $m \leftarrow \beta_1 m + (1 - \beta_1)\,g$ ; $v \leftarrow \beta_2 v + (1 - \beta_2)\,g^2$ ; $w \leftarrow w - \eta\,\frac{m / (1 - \beta_1^t)}{\sqrt{v / (1 - \beta_2^t)} + \epsilon}$. `state`, un dictionnaire vide au premier appel, garde ce qui sert d'un pas à l'autre. **[0,7]**
c) `train_epoch(params, X, y, state, lr, batch_size, max_norm, rng)` fait une epoch : `order = rng.permutation(len(X))`, puis les mini-batches de `batch_size` exemples dans cet ordre (le dernier peut être plus petit) ; pour chacun, `loss_and_grads`, le clipping, puis `adamw_step` (les autres réglages par défaut). Elle rend la loss moyenne de l'epoch par **exemple** (chaque loss de mini-batch pesée par son nombre d'exemples), un `float`. **[0,5]**
d) La cellule d'essai entraîne une epoch avec `LR`, `BATCH` et `MAX_NORM`, et `np.random.default_rng(49)` : reporte sur ta feuille $[\text{loss}, \text{accuracy}]$ (4 et 3 décimales). **[0,3]**

In [ ]:
def clip_by_global_norm(grads, max_norm):
    """(clipped, norm): new arrays scaled by min(1, max_norm / (norm + 1e-6)), and the global norm; CP4.9 a)."""
    norm = float(np.sqrt(sum(np.sum(g ** 2) for g in grads)))
    coef = min(1.0, max_norm / (norm + 1e-6))
    return [g * coef for g in grads], norm                     # g * coef is a new array: grads stay intact


def adamw_step(params, grads, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step, in place on the arrays of params, like torch.optim.AdamW; state is a dict; CP4.9 b)."""
    beta1, beta2 = betas
    if not state:                                              # first call: t = 0 and zero moments
        state["t"] = 0
        state["m"] = [np.zeros_like(p) for p in params]
        state["v"] = [np.zeros_like(p) for p in params]
    state["t"] += 1
    t = state["t"]
    for p, g, m, v in zip(params, grads, state["m"], state["v"]):
        p *= 1 - lr * weight_decay                             # decoupled weight decay, before the Adam step
        m *= beta1
        m += (1 - beta1) * g
        v *= beta2
        v += (1 - beta2) * g * g
        p -= lr * (m / (1 - beta1 ** t)) / (np.sqrt(v / (1 - beta2 ** t)) + eps)


def train_epoch(params, X, y, state, lr, batch_size, max_norm, rng):
    """One epoch in the order rng.permutation(len(X)); returns the mean loss per example, a float; CP4.9 c)."""
    order = rng.permutation(len(X))
    total = 0.0
    for start in range(0, len(X), batch_size):
        idx = order[start:start + batch_size]
        loss, grads = loss_and_grads(params, X[idx], y[idx])
        grads, _ = clip_by_global_norm(grads, max_norm)
        adamw_step(params, grads, state, lr)
        total += loss * len(idx)                               # weighted: the last mini-batch has 48 images
    return total / len(X)


params_9, state_9 = init_params(), {}
loss_9 = train_epoch(params_9, X_F, Y_F, state_9, LR, BATCH, MAX_NORM, np.random.default_rng(49))
acc_9 = accuracy(params_9, X_F, Y_F)
print(f"one epoch ({state_9['t']} steps): mean loss {loss_9:.4f} · accuracy {acc_9:.4f}")
try:                                                           # the same epoch with PyTorch, as a cross-check
    import torch
    W_t, b_t = (torch.tensor(a, requires_grad=True) for a in init_params())
    opt_t = torch.optim.AdamW([W_t, b_t], lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
    order_t = np.random.default_rng(49).permutation(len(X_F))
    X_t, Y_t = torch.from_numpy(X_F), torch.from_numpy(Y_F)
    for start in range(0, len(X_F), BATCH):
        idx = torch.from_numpy(order_t[start:start + BATCH])
        loss_t = torch.nn.functional.cross_entropy(X_t[idx] @ W_t + b_t, Y_t[idx])
        opt_t.zero_grad()
        loss_t.backward()
        torch.nn.utils.clip_grad_norm_([W_t, b_t], MAX_NORM)
        opt_t.step()
    gap_t = max(np.abs(W_t.detach().numpy() - params_9[0]).max(), np.abs(b_t.detach().numpy() - params_9[1]).max())
    print(f"torch.optim.AdamW + clip_grad_norm_ on the same mini-batches: largest gap {gap_t:.1e}")
except ImportError:
    print("(PyTorch is not installed: no cross-check)")

In [ ]:
def variant_9(*, clip=True, weighted=True, coupled=False, bias_correction=True, decay_after=False):
    """[loss, accuracy] of the epoch of CP4.9 written another (wrong) way."""
    params, state = init_params(), {"t": 0, "m": [np.zeros((784, 10)), np.zeros(10)], "v": [np.zeros((784, 10)), np.zeros(10)]}
    order = np.random.default_rng(49).permutation(len(X_F))
    losses, sizes = [], []
    for start in range(0, len(X_F), BATCH):
        idx = order[start:start + BATCH]
        loss, grads = loss_and_grads(params, X_F[idx], Y_F[idx])
        if clip:
            grads, _ = clip_by_global_norm(grads, MAX_NORM)
        state["t"] += 1
        t = state["t"]
        for p, g, m, v in zip(params, grads, state["m"], state["v"]):
            if coupled:
                g = g + 0.01 * p
            elif not decay_after:
                p *= 1 - LR * 0.01
            m *= 0.9
            m += 0.1 * g
            v *= 0.999
            v += 0.001 * g * g
            m_hat, v_hat = (m / (1 - 0.9 ** t), v / (1 - 0.999 ** t)) if bias_correction else (m, v)
            p -= LR * m_hat / (np.sqrt(v_hat) + 1e-8)
            if decay_after:
                p *= 1 - LR * 0.01
        losses.append(loss)
        sizes.append(len(idx))
    mean = np.average(losses, weights=sizes) if weighted else float(np.mean(losses))
    return [float(mean), accuracy(params, X_F, Y_F)]


wb.record("CP4.9d", [loss_9, acc_9], decimals=4, mistakes={
    "le clipping n'est pas appliqué : chaque gradient passe par clip_by_global_norm avant le pas d'AdamW":
    variant_9(clip=False),
    "la loss moyenne doit peser chaque mini-batch par son nombre d'exemples (le dernier n'en a que 48)":
    variant_9(weighted=False),
    "c'est le weight decay couplé d'Adam (λw ajouté au gradient) : AdamW multiplie w par (1 − ηλ), à part":
    variant_9(coupled=True),
    "les moments ne sont pas corrigés du biais : divise m par 1 − β₁ᵗ et v par 1 − β₂ᵗ":
    variant_9(bias_correction=False)})

## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [ ]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math

import numpy as np

# CP4.1
LAYERS_1 = [20, 64, 32, 3]
PARAMS_1 = sum(a * b + b for a, b in zip(LAYERS_1[:-1], LAYERS_1[1:]))
SOFTMAX_1 = math.exp(2) / (math.exp(2) + 2)

# CP4.2
X_2, Y_2 = np.array([2.0, -1.0]), 2.0
W1_2, B1_2 = np.array([[1.0, -0.5], [0.5, 1.0]]), np.array([-0.5, 0.0])
W2_2, B2_2 = np.array([0.75, -1.0]), 0.5


def forward_2(W1, b1, W2, b2, relu=True):
    z = X_2 @ W1 + b1
    h = np.maximum(z, 0) if relu else z
    return z, h, float(h @ W2 + b2)


Z_2, H_2, YHAT_2 = forward_2(W1_2, B1_2, W2_2, B2_2)
LOSS_2 = 0.5 * (YHAT_2 - Y_2) ** 2
DELTA_2 = YHAT_2 - Y_2
DW2_2, DB2_2 = H_2 * DELTA_2, DELTA_2
DH_2 = DELTA_2 * W2_2
DZ_2 = DH_2 * (Z_2 > 0)
DW1_2 = np.outer(X_2, DZ_2)
ETA_2 = 0.1
W1N_2, B1N_2, W2N_2, B2N_2 = W1_2 - ETA_2 * DW1_2, B1_2 - ETA_2 * DZ_2, W2_2 - ETA_2 * DW2_2, B2_2 - ETA_2 * DB2_2


def loss_after_2(W1, b1, W2, b2):
    return 0.5 * (forward_2(W1, b1, W2, b2)[2] - Y_2) ** 2


# CP4.3


def momentum_3(dampening=0.0):
    w, b, history = 1.0, None, []
    for g in (2.0, 1.0):
        b = g if b is None else 0.9 * b + (1 - dampening) * g
        w -= 0.1 * b
        history.append((b, w))
    return history


def adam_3(steps=2, weight_decay=0.0, decoupled=True, bias_correction=True, decay_after=False):
    w, m, v, out = 1.0, 0.0, 0.0, []
    for t, g in enumerate((2.0, 1.0)[:steps], start=1):
        if weight_decay and not decoupled:
            g = g + weight_decay * w
        if weight_decay and decoupled and not decay_after:
            w *= 1 - 0.01 * weight_decay
        m = 0.9 * m + 0.1 * g
        v = 0.999 * v + 0.001 * g * g
        m_hat, v_hat = (m / (1 - 0.9 ** t), v / (1 - 0.999 ** t)) if bias_correction else (m, v)
        w -= 0.01 * m_hat / math.sqrt(v_hat)
        if weight_decay and decoupled and decay_after:
            w *= 1 - 0.01 * weight_decay
        out.append((w, m_hat, v_hat, m, v))
    return out


# CP4.4
XD_4, U_4, P_4 = np.array([2.0, 4.0, 6.0, 8.0]), np.array([0.1, 0.6, 0.3, 0.9]), 0.25
DROP_4 = XD_4 * (U_4 >= P_4) / (1 - P_4)
XB_4, GAMMA_4, BETA_4, RHO_4 = np.array([2.0, 6.0, 8.0, 12.0]), 0.5, -1.0, 0.1
MEAN_4, VAR_4, UNBIASED_4 = XB_4.mean(), XB_4.var(), XB_4.var(ddof=1)
RUN_MEAN_4, RUN_VAR_4 = (1 - RHO_4) * 0 + RHO_4 * MEAN_4, (1 - RHO_4) * 1 + RHO_4 * UNBIASED_4


def bn_train_4(x, var):
    return GAMMA_4 * (x - MEAN_4) / math.sqrt(var) + BETA_4


def bn_eval_4(x, mean=RUN_MEAN_4, var=RUN_VAR_4):
    return GAMMA_4 * (x - mean) / math.sqrt(var) + BETA_4


# CP4.5
L_5 = min(n for n in range(1, 100) if 0.25 ** n < 1e-6)

# CP4.6
PARAMS_6 = 784 * 1024 + 1024 + 1024 * 1024 + 1024 + 1024 * 10 + 10
STEPS_6 = math.ceil(60000 / 256) * 10

# CP4.12
P_12 = 0.7
TRUE_12 = [0.7, 0.5, 0.9]
XR_12, YR_12 = np.array([1.0, 2.0, 3.0]), np.array([2.0, 3.0, 7.0])
FLIPPERS_12, TEST_12 = np.array([182.0, 191.0, 197.0, 210.0]), 216.0
WITH_TEST_12 = np.append(FLIPPERS_12, TEST_12)

**CP4.1 — Questions flash**

In [ ]:
wb.record("CP4.1a", PARAMS_1, mistakes={"compte aussi les biais : un par neurone, dans chaque couche sauf l'entrée": PARAMS_1 - (64 + 32 + 3)})
wb.record("CP4.1b", "(16, 32)", mistakes={"la forme d'une sortie est (nombre d'exemples, nombre de neurones), dans cet ordre": "(32, 16)",
                     "c'est la sortie de la première couche cachée : on demande celle de la deuxième": "(16, 64)",
                     "la forme d'une sortie garde l'axe des exemples": "(32,)"})
wb.record("CP4.1c", [0.0, 0.5, 1.0, -0.2], decimals=2, mistakes={"tanh′(z) = 1 − tanh²(z) : en 0, la pente vaut 1": [0.0, 0.5, 0.0, -0.2],
                     "la leaky ReLU de l'énoncé a une pente de 0,1, pas 0,01": [0.0, 0.5, 1.0, -0.02],
                     "ReLU(z) = max(0, z) : une valeur négative donne 0": [-1.5, 0.5, 1.0, -0.2],
                     "σ(0) = 1 / (1 + e⁰)": [0.0, 1.0, 1.0, -0.2]})
wb.record("CP4.1d", SOFTMAX_1, decimals=3, mistakes={"le dénominateur somme les exponentielles des TROIS logits, e² + e⁰ + e⁰": math.exp(2) / (math.exp(2) + 1),
                     "c'est la probabilité d'un des deux autres logits : on demande la première": 1 / (math.exp(2) + 2),
                     "la softmax passe par les exponentielles : ce n'est pas 2 / (2 + 0 + 0)": 1.0})
wb.record("CP4.1e", 128, mistakes={"running_mean et running_var sont des buffers, pas des paramètres appris : il reste γ et β": 256,
                     "γ ET β : deux vecteurs de 64 valeurs": 64, "la batchnorm de PyTorch a bien des paramètres appris": 0})
wb.record("CP4.1f", 8 * (3 * 3 * 3) + 8, mistakes={"compte aussi les biais, un par filtre": 8 * 27,
                     "un filtre couvre les 3 canaux d'entrée : 3 × 3 × 3 poids": 8 * 9 + 8})

**CP4.2 — Passe avant, rétropropagation et un pas de SGD**

In [ ]:
wb.record("CP4.2a", Z_2.tolist(), decimals=4, mistakes={"n'oublie pas les biais de la couche cachée": (X_2 @ W1_2).tolist(),
                     "dans la convention z = x W, la colonne k de W porte les poids du neurone k : lis W par colonnes": (X_2 @ W1_2.T + B1_2).tolist()})
wb.record("CP4.2b", [YHAT_2, LOSS_2], decimals=5, mistakes={"la loss de l'énoncé a un facteur ½ : L = ½ (ŷ − y)²": [YHAT_2, (YHAT_2 - Y_2) ** 2],
                     "un neurone caché à ReLU de somme négative sort 0, pas sa somme": [forward_2(W1_2, B1_2, W2_2, B2_2, relu=False)[2], 0.5 * (forward_2(W1_2, B1_2, W2_2, B2_2, relu=False)[2] - Y_2) ** 2]})
wb.record("CP4.2c", DELTA_2, decimals=4, mistakes={"avec le facteur ½, la dérivée de ½ (ŷ − y)² est ŷ − y, sans le 2": 2 * DELTA_2})
wb.record("CP4.2d", [*DW2_2.tolist(), DB2_2], decimals=4, mistakes={"le gradient d'un poids de sortie est δ × la SORTIE du neurone caché (après la ReLU), pas sa somme": [*(Z_2 * DELTA_2).tolist(), DB2_2],
                     "le biais de sortie reçoit δ × 1": [*DW2_2.tolist(), 0.0]})
wb.record("CP4.2e", DZ_2.tolist(), decimals=4, mistakes={"n'oublie pas le facteur ReLU′(z) : 0 pour un neurone de somme négative": DH_2.tolist()})
wb.record("CP4.2f", DW1_2.tolist(), decimals=4, mistakes={"range le gradient comme W(1) : ligne j = l'entrée x_j, colonne k = le neurone k": DW1_2.T.tolist(),
                     "n'oublie pas le facteur ReLU′(z) du neurone 2": np.outer(X_2, DH_2).tolist()})
wb.record("CP4.2g", W1N_2.tolist(), decimals=5, mistakes={"on descend le gradient : w ← w − η ∂L/∂w": (W1_2 + ETA_2 * DW1_2).tolist(),
                     "c'est un pas de learning rate 1 : l'énoncé prend η = 0,1": (W1_2 - DW1_2).tolist()})
wb.record("CP4.2h", loss_after_2(W1N_2, B1N_2, W2N_2, B2N_2), decimals=5, mistakes={"le pas met à jour TOUS les paramètres, la couche de sortie comprise": loss_after_2(W1N_2, B1N_2, W2_2, B2_2),
                     "c'est la loss d'avant le pas": LOSS_2,
                     "les biais aussi font un pas": loss_after_2(W1N_2, B1_2, W2N_2, B2_2)})

**CP4.3 — Momentum, Adam et AdamW**

In [ ]:
wb.record("CP4.3a", momentum_3()[1][0], decimals=4, mistakes={"la convention de PyTorch n'a pas de facteur (1 − μ) : b ← μ b + g": momentum_3(dampening=0.9)[1][0],
                     "relis la convention : la vitesse du deuxième pas part de b₁ = g₁, pas de 0": 1.0,
                     "tu as calculé une moyenne mobile, b ← μ b + (1 − μ) g ; la convention de torch.optim.SGD n'a pas de facteur (1 − μ), et b₁ = g₁": 0.28})
wb.record("CP4.3b", momentum_3()[1][1], decimals=4, mistakes={"le premier pas a déjà déplacé w : w₂ = w₁ − η b₂": 1 - 0.1 * momentum_3()[1][0],
                     "la convention de PyTorch n'a pas de facteur (1 − μ) : b ← μ b + g": momentum_3(dampening=0.9)[1][1],
                     "tu as calculé une moyenne mobile, b ← μ b + (1 − μ) g ; la convention de torch.optim.SGD n'a pas de facteur (1 − μ), et b₁ = g₁": 1 - 0.1 * 0.2 - 0.1 * 0.28})
wb.record("CP4.3c", adam_3(1)[0][0], decimals=5, mistakes={"corrige m et v du biais : divise par 1 − β₁ et 1 − β₂ au premier pas": adam_3(1, bias_correction=False)[0][0]})
wb.record("CP4.3d", [adam_3()[1][1], adam_3()[1][2]], decimals=3, mistakes={"ce sont les moments NON corrigés : divise par 1 − β₁² et 1 − β₂²": [adam_3()[1][3], adam_3()[1][4]],
                     "au deuxième pas, t = 2 : la correction divise par 1 − β², pas par 1 − β": [adam_3()[1][3] / 0.1, adam_3()[1][4] / 0.001]})
wb.record("CP4.3e", adam_3()[1][0], decimals=5, mistakes={"corrige m et v du biais à chaque pas": adam_3(bias_correction=False)[1][0]})
wb.record("CP4.3f", adam_3(1, weight_decay=0.1)[0][0], decimals=5, mistakes={"AdamW multiplie w par (1 − ηλ) AVANT le pas d'Adam, comme torch.optim.AdamW": adam_3(1, weight_decay=0.1, decay_after=True)[0][0],
                     "le facteur est 1 − ηλ, avec le learning rate : 1 − 0,01 × 0,1": 0.9 * 1 - 0.01,
                     "c'est le weight decay couplé d'Adam : AdamW le découple": adam_3(1, weight_decay=0.1, decoupled=False)[0][0]})
wb.record("CP4.3g", adam_3(1, weight_decay=0.1, decoupled=False)[0][0], decimals=5, mistakes={"c'est le weight decay découplé d'AdamW : Adam(weight_decay) ajoute λw au GRADIENT, avant la normalisation": adam_3(1, weight_decay=0.1)[0][0]})

**CP4.4 — Dropout et batchnorm**

In [ ]:
wb.record("CP4.4a", DROP_4.tolist(), decimals=3, mistakes={"le dropout INVERSÉ divise les valeurs gardées par 1 − p": (XD_4 * (U_4 >= P_4)).tolist(),
                     "une valeur est gardée quand u ≥ p, et mise à zéro sinon": (XD_4 * (U_4 < P_4) / (1 - P_4)).tolist(),
                     "on divise par la probabilité de GARDER une valeur, 1 − p": (XD_4 * (U_4 >= P_4) / P_4).tolist()})
wb.record("CP4.4b", XD_4.tolist(), decimals=3, mistakes={"en évaluation, le dropout inversé ne fait rien : il rend son entrée": (XD_4 / (1 - P_4)).tolist(),
                     "c'est le dropout « non inversé » du livre : PyTorch multiplie déjà en entraînement, et ne touche à rien en évaluation": (XD_4 * (1 - P_4)).tolist(),
                     "en évaluation, aucune valeur n'est mise à zéro": DROP_4.tolist()})
wb.record("CP4.4c", float(DROP_4.mean()), decimals=3, mistakes={"c'est la moyenne de x : on demande celle des quatre valeurs de a)": float(XD_4.mean())})
wb.record("CP4.4d", [MEAN_4, VAR_4], decimals=4, mistakes={"la batchnorm normalise avec la variance BIAISÉE (diviser par m)": [MEAN_4, UNBIASED_4],
                     "c'est l'écart-type : on demande la variance": [MEAN_4, math.sqrt(VAR_4)]})
wb.record("CP4.4e", bn_train_4(12.0, VAR_4), decimals=4, mistakes={"la normalisation utilise la variance biaisée du mini-batch": bn_train_4(12.0, UNBIASED_4),
                     "applique ensuite γ et β : y = γ x̂ + β": (12.0 - MEAN_4) / math.sqrt(VAR_4)})
wb.record("CP4.4f", [RUN_MEAN_4, RUN_VAR_4], decimals=4, mistakes={"la variance glissante se met à jour avec la variance NON biaisée du mini-batch": [RUN_MEAN_4, (1 - RHO_4) * 1 + RHO_4 * VAR_4],
                     "la mise à jour garde 1 − ρ = 0,9 de l'ancienne valeur et prend ρ = 0,1 du mini-batch": [RHO_4 * 0 + (1 - RHO_4) * MEAN_4, RHO_4 * 1 + (1 - RHO_4) * UNBIASED_4],
                     "la couche est neuve : sa variance glissante part de 1, pas de 0": [RUN_MEAN_4, RHO_4 * UNBIASED_4],
                     "la variance glissante mélange l'ancienne valeur et celle du mini-batch, comme la moyenne glissante": [RUN_MEAN_4, UNBIASED_4]})
wb.record("CP4.4g", bn_eval_4(7.0), decimals=4, mistakes={"en évaluation, la batchnorm utilise ses statistiques GLISSANTES, pas celles d'un mini-batch": BETA_4,
                     "applique ensuite γ et β : y = γ x̂ + β": (7.0 - RUN_MEAN_4) / math.sqrt(RUN_VAR_4),
                     "la variance glissante se met à jour avec la variance NON biaisée du mini-batch, comme en f)": bn_eval_4(7.0, var=(1 - RHO_4) * 1 + RHO_4 * VAR_4)})

**CP4.5 — Le gradient à travers des sigmoïdes empilées (c)**

In [ ]:
wb.record("CP4.5c", L_5, fractional="on demande un nombre de couches", mistakes={"calcule (1/4)⁹ : la borne dépasse encore 10⁻⁶": 9,
                     "la pente de la sigmoïde est au plus 1/4, pas 1/2": min(n for n in range(1, 100) if 0.5 ** n < 1e-6)})

**CP4.6 — Fermi : paramètres et mémoire**

In [ ]:
wb.record("CP4.6a", PARAMS_6 / 1e6, decimals=1, mistakes={"en millions : divise le nombre de paramètres par 10⁶": PARAMS_6 / 1e5})
wb.record("CP4.6b", round(PARAMS_6 * 4 * 4 / 1e6), fractional="arrondis à l'entier", mistakes={"ce sont les paramètres seuls : ajoute les gradients et les deux moments d'AdamW": round(PARAMS_6 * 4 / 1e6),
                     "paramètres et gradients seulement : AdamW garde aussi ses deux moments": round(PARAMS_6 * 4 * 2 / 1e6),
                     "il manque un des quatre tableaux de même taille (paramètres, gradients, m, v)": round(PARAMS_6 * 4 * 3 / 1e6),
                     "un mégaoctet vaut 10⁶ octets ici, pas 2²⁰": round(PARAMS_6 * 16 / 2 ** 20)})
wb.record("CP4.6c", 2, fractional="un rapport entier", mistakes={"compte les tableaux de la taille des paramètres : 4 pour AdamW, 2 sans moments": 4,
                     "AdamW garde DEUX moments par paramètre, m et v : quatre tableaux contre deux": 3})
wb.record("CP4.6d", STEPS_6, fractional="un nombre de mises à jour", mistakes={"le dernier mini-batch, incomplet, est aussi une mise à jour : arrondis au-dessus": (60000 // 256) * 10,
                     "c'est une seule epoch : on en demande 10": STEPS_6 // 10})

**CP4.7 — Les lettres des quatre lignes fautives (a)**

In [ ]:
wb.record("CP4.7a", "DIKM", mistakes={"il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKM",
                     "il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKM",
                     "il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DIM",
                     "il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIK",
                     "une ligne de trop : la ligne (A) est correcte ; relis ce qu'elle fait": "ADIKM",
                     "une ligne de trop : la ligne (B) est correcte ; relis ce qu'elle fait": "BDIKM",
                     "une ligne de trop : la ligne (C) est correcte ; relis ce qu'elle fait": "CDIKM",
                     "une ligne de trop : la ligne (E) est correcte ; relis ce qu'elle fait": "DEIKM",
                     "une ligne de trop : la ligne (F) est correcte ; relis ce qu'elle fait": "DFIKM",
                     "une ligne de trop : la ligne (G) est correcte ; relis ce qu'elle fait": "DGIKM",
                     "une ligne de trop : la ligne (H) est correcte ; relis ce qu'elle fait": "DHIKM",
                     "une ligne de trop : la ligne (J) est correcte ; relis ce qu'elle fait": "DIJKM",
                     "une ligne de trop : la ligne (L) est correcte ; relis ce qu'elle fait": "DIKLM",
                     "une ligne de trop : la ligne (N) est correcte ; relis ce qu'elle fait": "DIKMN",
                     "une ligne de trop : la ligne (O) est correcte ; relis ce qu'elle fait": "DIKMO",
                     "une ligne de trop : la ligne (P) est correcte ; relis ce qu'elle fait": "DIKMP",
                     "une ligne de trop : la ligne (Q) est correcte ; relis ce qu'elle fait": "DIKMQ",
                     "la ligne (A) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "AIKM",
                     "la ligne (B) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "BIKM",
                     "la ligne (C) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "CIKM",
                     "la ligne (E) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "EIKM",
                     "la ligne (F) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "FIKM",
                     "la ligne (G) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "GIKM",
                     "la ligne (H) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "HIKM",
                     "la ligne (J) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IJKM",
                     "la ligne (L) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKLM",
                     "la ligne (N) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKMN",
                     "la ligne (O) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKMO",
                     "la ligne (P) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKMP",
                     "la ligne (Q) est correcte, et il manque une ligne fautive : une ligne crée à chaque epoch ce qui devrait vivre tout l'entraînement": "IKMQ",
                     "la ligne (A) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "ADKM",
                     "la ligne (B) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "BDKM",
                     "la ligne (C) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "CDKM",
                     "la ligne (E) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DEKM",
                     "la ligne (F) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DFKM",
                     "la ligne (G) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DGKM",
                     "la ligne (H) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DHKM",
                     "la ligne (J) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DJKM",
                     "la ligne (L) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKLM",
                     "la ligne (N) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKMN",
                     "la ligne (O) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKMO",
                     "la ligne (P) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKMP",
                     "la ligne (Q) est correcte, et il manque une ligne fautive : regarde ce que reçoit loss_fn, des logits ou des probabilités ?": "DKMQ",
                     "la ligne (A) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "ADIM",
                     "la ligne (B) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "BDIM",
                     "la ligne (C) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "CDIM",
                     "la ligne (E) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DEIM",
                     "la ligne (F) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DFIM",
                     "la ligne (G) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DGIM",
                     "la ligne (H) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DHIM",
                     "(J) et (K) sont bien dans le mauvais ordre, mais l'énoncé demande la ligne qui arrive trop tard": "DIJM",
                     "la ligne (L) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DILM",
                     "la ligne (N) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DIMN",
                     "la ligne (O) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DIMO",
                     "la ligne (P) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DIMP",
                     "la ligne (Q) est correcte, et il manque une ligne fautive : dans quel ordre doivent venir zero_grad, backward et step ?": "DIMQ",
                     "la ligne (A) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "ADIK",
                     "la ligne (B) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "BDIK",
                     "la ligne (C) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "CDIK",
                     "la ligne (E) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DEIK",
                     "la ligne (F) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DFIK",
                     "la ligne (G) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DGIK",
                     "la ligne (H) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DHIK",
                     "la ligne (J) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIJK",
                     "la ligne (L) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIKL",
                     "la ligne (N) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIKN",
                     "la ligne (O) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIKO",
                     "la ligne (P) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIKP",
                     "la ligne (Q) est correcte, et il manque une ligne fautive : que garde en mémoire la somme d'un tenseur qui a un grad_fn ?": "DIKQ"})

**CP4.8 — Lire les quatre paires de courbes**

In [ ]:
wb.record("CP4.8a", "A", mistakes={"un learning rate trop petit fait baisser la loss lentement et régulièrement : celle-ci saute et ne baisse pas": "B",
                     "l'overfitting suppose une loss d'entraînement qui baisse : regarde la courbe d'entraînement": "C",
                     "un entraînement sain fait baisser la loss : regarde l'échelle": "D",
                     "une fuite donne des losses basses et collées, pas des sauts": "E",
                     "eval() oublié laisse la loss d'entraînement baisser normalement : est-ce le cas ici ?": "F"})
wb.record("CP4.8b", "C", mistakes={"la loss d'entraînement baisse régulièrement jusqu'à presque 0 : le pas n'est pas trop grand": "A",
                     "la loss d'entraînement baisse vite : le pas n'est pas trop petit": "B",
                     "regarde la loss de validation après son minimum, et l'écart avec l'entraînement": "D",
                     "une fuite ferait suivre l'entraînement à la validation : ici elles s'écartent": "E",
                     "eval() oublié rend la validation bruitée et proche de l'entraînement : ici elle s'en écarte nettement": "F"})
wb.record("CP4.8c", "B", mistakes={"un pas trop grand fait sauter la loss : ici elle baisse à peine, sans un seul saut": "A",
                     "l'overfitting suppose une loss d'entraînement basse : où en est-elle après 30 epochs ?": "C",
                     "un entraînement sain aurait fait baisser la loss bien plus vite : relis l'échelle": "D",
                     "une fuite donne des losses BASSES : celles-ci sont encore près de ln 10": "E",
                     "eval() oublié n'empêche pas l'entraînement d'avancer : ici rien n'avance": "F"})
wb.record("CP4.8d", "D", mistakes={"la loss baisse régulièrement, sans saut : le pas n'est pas trop grand": "A",
                     "la loss baisse vite les premières epochs : le pas n'est pas trop petit": "B",
                     "la loss de validation remonte-t-elle à la fin ?": "C",
                     "une fuite donnerait des losses presque nulles et identiques : ce n'est pas le cas": "E",
                     "eval() oublié rendrait la validation bruitée : elle est ici régulière": "F"})

**CP4.12 — Parties I à III**

In [ ]:
wb.record("CP4.12a", -math.log(P_12), decimals=3, mistakes={"la cross-entropy ne regarde que la probabilité de la VRAIE classe : −ln 0,7": -(0.2 * math.log(0.2) + 0.1 * math.log(0.1) + 0.7 * math.log(0.7)),
                     "en nats : le logarithme népérien, pas log₁₀": -math.log10(P_12),
                     "c'est en bits : on la demande en nats (ln)": -math.log2(P_12)})
wb.record("CP4.12b", -math.log2(P_12), decimals=3, mistakes={"c'est en nats : en bits, log₂ (diviser par ln 2)": -math.log(P_12),
                     "en bits : log₂, pas log₁₀": -math.log10(P_12)})
wb.record("CP4.12c", float(np.mean([-math.log(q) for q in TRUE_12])), decimals=4, mistakes={"on fait la moyenne des −ln p, pas −ln de la moyenne des p": -math.log(np.mean(TRUE_12)),
                     "une moyenne, pas une somme": float(np.sum([-math.log(q) for q in TRUE_12]))})
wb.record("CP4.12d", [float(XR_12 @ YR_12 / (XR_12 @ XR_12)), float(XR_12 @ YR_12 / (XR_12 @ XR_12 + 7))], decimals=3, mistakes={"la pénalité α w² s'ajoute au dénominateur : w = Σxy / (Σx² + α)": [float(XR_12 @ YR_12 / (XR_12 @ XR_12)), float((XR_12 @ YR_12 - 7) / (XR_12 @ XR_12))],
                     "sans ordonnée à l'origine, w = Σxy / Σx², pas le rapport des moyennes": [float(YR_12.mean() / XR_12.mean()), float(XR_12 @ YR_12 / (XR_12 @ XR_12 + 7))]})
wb.record("CP4.12e", float((TEST_12 - FLIPPERS_12.mean()) / FLIPPERS_12.std()), decimals=3, mistakes={"le scaler apprend sur l'entraînement SEUL : le manchot de test n'entre pas dans la moyenne": float((TEST_12 - WITH_TEST_12.mean()) / WITH_TEST_12.std()),
                     "StandardScaler divise par n, pas par n − 1": float((TEST_12 - FLIPPERS_12.mean()) / FLIPPERS_12.std(ddof=1))})

**CP4.13 — Prédire ce que font cinq extraits PyTorch**

In [ ]:
wb.record("CP4.13a", "B", mistakes={"le graphe est reconstruit à chaque tour : que fait backward() de la valeur déjà présente dans .grad ?": "A",
                     "chaque tour recalcule (2 * x).sum() : un graphe neuf, qui peut être rétropropagé": "C",
                     "x a requires_grad=True : backward() remplit x.grad": "D"})
wb.record("CP4.13b", "C", mistakes={"en mode évaluation, le dropout ne tire plus de masque": "A",
                     "PyTorch fait un dropout INVERSÉ : il ne multiplie rien par 1 − p en évaluation": "B",
                     "en mode évaluation, le dropout ne met plus rien à zéro": "D"})
wb.record("CP4.13c", "C", mistakes={"une couche neuve est en mode entraînement : avec un seul exemple, que vaut la variance du mini-batch ?": "A",
                     "une couche neuve est en mode entraînement : elle normalise avec les statistiques du mini-batch": "B",
                     "PyTorch vérifie la taille du mini-batch avant de calculer quoi que ce soit": "D"})
wb.record("CP4.13d", "A", mistakes={"PyTorch range le poids de nn.Linear en (n_out, n_in), la transposée de la convention du workbook": "B",
                     "le poids est une matrice, pas un vecteur aplati": "C",
                     "c'est la forme du biais : on demande celle du poids": "D"})
wb.record("CP4.13e", "D", mistakes={"des logits nuls donnent une softmax uniforme : la probabilité de la bonne classe vaut 1/4": "A",
                     "CrossEntropyLoss rend −ln de la probabilité de la bonne classe, en moyenne : pas la probabilité elle-même": "B",
                     "CrossEntropyLoss attend des logits : des zéros sont des logits parfaitement valables": "C"})